## 맛있는 절약 : 평가용 dataset 생성
1. find_keyIngredients_tasty : (사전 생성) 레시피의 핵심 맛과 핵심 재료를 판단
2. generate_food_group_radio : (사전 생성) 레시피의 식품군을 판별
3. fridge_recipe_transform : (기능) 냉장고 재료 대체 
4. simple_recipe_transform : (기능) 레시피 간소화 
5. balance_nutrition : (기능) 영양 맞춤형 레시피 

## 일반적인 평가용 dataset 생성 case
- 검색한 것이 질문과 관련이 있는가?
- 답변이 질문과 관련이 있는가?
- 답변을 검색한 문서 안에서 답변한 것인가?(할루시네이션 check)

## 한계
검색한 결과에 대한 정확한 기준 데이터(Ground truth) 구축이 사실상 어려움
따라서, 검색한 결과에 대한 Ground truth가 존재한다면 모두 데이터셋으로 활용하고, 아니라면 질문과 답변만으로 데이터셋을 구축하여 활용함
⇒ 우리는 레시피 생성시에 RAG가 없기 때문에 질문과 답변만으로 데이터셋을 구축해서 활용함

## 레시피 dataset
spicy_level, type_key, method_key, servings를 기준으로 각각 중복되지 않은 행만 추출 

In [131]:
import pandas as pd
df_recipeinfo = pd.read_csv("data/diverse_recipes_dataset.csv")
df_recipeinfo.describe()

,_id,title,type_key,method_key,servings,cooking_time,difficulty,ingredients,cooking_steps,tips,recipe_type,spicy_level
count,35,35,35,35,34,33,33,35,35,35,35,35
unique,35,35,12,13,5,7,2,35,35,17,35,6
top,67610699846f9e5eb975e532,연어샐러드,반찬,끓이기,1인분,30분 이내,초급,"['연어(150g)', '레몬(1/4개)', '발사믹식초(50g)', '어린잎채소(30g)', '후춧가루(0.01g)', '올리브오일(20g)']","['1. 연어는 깍둑썰기한다.', '2. 썰어 놓은 연어는 후춧가루와 레몬으로 마리네이드한다.', '3. 어린잎은 찬물에 담궈둔다.', '4. 담궈 놓은 어린잎을 체에 받쳐 물기를 뺀다.', '5. 레몬과 올리브오일을 섞는다.', '6. ?번에 발사믹소스를 넣고 연어 샐러드 양념을 만들고, 접시에 연어와 물기를 뺀 어린잎을 담는다.']",[],"['고단백', '저칼로리', '해산물', '샐러드']",매운맛없음
freq,1,1,10,10,12,14,29,1,1,19,1,23


In [132]:
df_userinfo = pd.read_csv('data/persona_csv.csv')
df_userinfo.describe()

,persona,user_allergy_ingredients,user_dislike_ingredients,user_spicy_level,user_cooking_level,user_owned_ingredients,user_basic_seasoning,must_use_ingredients
count,10,10,10,10,10,10,10,10
unique,10,4,10,5,3,10,10,8
top,유미,[],[],3단계,초급,"['닭가슴살', '두부', '브로콜리']","['소금', '후추', '올리브유']",['닭가슴살']
freq,1,7,1,3,5,1,1,2


In [133]:
import pandas as pd

# Recipe와 User 정보를 균등하게 결합하여 JSON 형식으로 변환
def create_combined_dataset(recipeinfo_df, userinfo_df):
    combined_data = []
    len_userinfo = len(userinfo_df)  # UserInfo 데이터 수
    
    for idx, recipe_row in recipeinfo_df.iterrows():
        user_idx = idx % len_userinfo  # user_info를 균등하게 분배하기 위한 인덱스
        user_row = userinfo_df.iloc[user_idx]
        
        combined_entry = {
            "user_info": {
                "user_allergy_ingredients": eval(user_row["user_allergy_ingredients"]),
                "user_dislike_ingredients": eval(user_row["user_dislike_ingredients"]),
                "user_spicy_level": user_row["user_spicy_level"],
                "user_cooking_level": user_row["user_cooking_level"],
                "user_owned_ingredients": eval(user_row["user_owned_ingredients"]),
                "user_basic_seasoning": eval(user_row["user_basic_seasoning"]),
                "must_use_ingredients": eval(user_row["must_use_ingredients"]),
            },
            "recipe_info": {
                "_id": recipe_row["_id"],
                "title": recipe_row["title"],
                "type_key": recipe_row["type_key"],
                "method_key": recipe_row["method_key"],
                "servings": recipe_row["servings"],
                "cooking_time": recipe_row["cooking_time"] if not pd.isna(recipe_row["cooking_time"]) else "",
                "difficulty": recipe_row["difficulty"] if not pd.isna(recipe_row["difficulty"]) else "",
                "ingredients": eval(recipe_row["ingredients"]),
                "cooking_steps": eval(recipe_row["cooking_steps"]),
                "tips": eval(recipe_row["tips"]),
            },
        }
        combined_data.append(combined_entry)
    
    return combined_data

# 데이터셋 생성
combined_dataset = create_combined_dataset(df_recipeinfo, df_userinfo)

# JSON 데이터셋 저장 (옵션)
import json
with open('data/combined_dataset.json', 'w', encoding='utf-8') as f:
    json.dump(combined_dataset, f, ensure_ascii=False, indent=4)

print("데이터셋 생성 완료. 첫 번째 데이터:")
print(combined_dataset[0])
print("갯수:", len(combined_dataset))


데이터셋 생성 완료. 첫 번째 데이터:
{'user_info': {'user_allergy_ingredients': [], 'user_dislike_ingredients': [], 'user_spicy_level': '3단계', 'user_cooking_level': '중급', 'user_owned_ingredients': ['닭가슴살', '두부', '브로콜리'], 'user_basic_seasoning': ['소금', '후추', '올리브유'], 'must_use_ingredients': ['닭가슴살']}, 'recipe_info': {'_id': '67610699846f9e5eb975e532', 'title': '연어샐러드', 'type_key': '샐러드', 'method_key': '회', 'servings': '1인분', 'cooking_time': '', 'difficulty': '', 'ingredients': ['연어(150g)', '레몬(1/4개)', '발사믹식초(50g)', '어린잎채소(30g)', '후춧가루(0.01g)', '올리브오일(20g)'], 'cooking_steps': ['1. 연어는 깍둑썰기한다.', '2. 썰어 놓은 연어는 후춧가루와 레몬으로 마리네이드한다.', '3. 어린잎은 찬물에 담궈둔다.', '4. 담궈 놓은 어린잎을 체에 받쳐 물기를 뺀다.', '5. 레몬과 올리브오일을 섞는다.', '6. ?번에 발사믹소스를 넣고 연어 샐러드 양념을 만들고, 접시에 연어와 물기를 뺀 어린잎을 담는다.'], 'tips': ['발사믹소스에 레몬과 올리브오일을 섞어 별도의 소금을 넣지 않을 수 있어요.']}}
갯수: 35


In [1]:
import pandas as pd

# JSON 파일 경로
file_path = 'data/combined_dataset.json'

# JSON 파일을 DataFrame으로 불러오기
df = pd.read_json(file_path)

for i in range(len(df)):
    print(df.user_info[i])


{'user_allergy_ingredients': [], 'user_dislike_ingredients': [], 'user_spicy_level': '3단계', 'user_cooking_level': '중급', 'user_owned_ingredients': ['닭가슴살', '두부', '브로콜리'], 'user_basic_seasoning': ['소금', '후추', '올리브유'], 'must_use_ingredients': ['닭가슴살']}
{'user_allergy_ingredients': [], 'user_dislike_ingredients': ['토마토', '오이'], 'user_spicy_level': '2단계', 'user_cooking_level': '초급', 'user_owned_ingredients': ['계란', '감자', '스팸'], 'user_basic_seasoning': ['간장', '설탕', '후추'], 'must_use_ingredients': ['계란']}
{'user_allergy_ingredients': [], 'user_dislike_ingredients': ['고수'], 'user_spicy_level': '4단계', 'user_cooking_level': '고급', 'user_owned_ingredients': ['버섯', '파스타면', '레몬'], 'user_basic_seasoning': ['올리브유', '소금', '파프리카 가루'], 'must_use_ingredients': ['버섯']}
{'user_allergy_ingredients': [], 'user_dislike_ingredients': ['치즈'], 'user_spicy_level': '3단계', 'user_cooking_level': '초급', 'user_owned_ingredients': ['양파', '라면', '김치'], 'user_basic_seasoning': ['소금', '고춧가루', '간장'], 'must_use_ingredients': ['

## 작은 datatset 생성

In [2]:
df[:1]

,user_info,recipe_info
0,"{'user_allergy_ingredients': [], 'user_dislike...","{'_id': '67610699846f9e5eb975e532', 'title': '..."


In [ ]:
import json
with open('data/small_combined_dataset.json', 'w', encoding='utf-8') as f:
    json.dump(combined_dataset[:2], f, ensure_ascii=False, indent=4)

### dataset에 find_keyIngredients_tasty 결과 추가

## llm 기능 실행

In [3]:
from langchain_teddynote import logging
# set_enable=False 로 지정하면 추적을 하지 않습니다.
logging.langsmith("랭체인 튜토리얼 프로젝트", set_enable=False)

LangSmith 추적을 하지 않습니다.


## 3개 행으로 연습

In [9]:
import sys
import pandas as pd
sys.path.append('../')  # 상위 디렉토리의 src 폴더를 경로에 추가
from src.recipe_change_origin import generate_recipe

df = pd.read_json('data/combined_dataset.json')
df_cp = df[:3].copy().reset_index(drop=True)

df_cp['generation'] = None  # 열 초기화

for i in range(len(df_cp)):
    df_cp.at[i, 'generation'] = await generate_recipe(df_cp.iloc[i].recipe_info, df.iloc[i].user_info, 1)

df_cp

2024-12-23 00:22:35,114 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 00:22:35,115 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 00:22:35,120 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 00:22:35,121 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 00:22:35,122 - recipe_logger - INFO - LLM 레시피 생성 중...
2024-12-23 00:22:54,044 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 00:22:54,061 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 00:22:54,062 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 00:22:54,063 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 00:22:54,065 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 00:22:54,066 - recipe_logger - INFO - LLM 레시피 생성 중...
2024-12-23 00:23:15,890 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 00:23:15,905 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 00:23:15,906 - recipe_logger - INFO - langfuse pr

,user_info,recipe_info,generation
0,"{'user_allergy_ingredients': [], 'user_dislike...","{'_id': '67610699846f9e5eb975e532', 'title': '...",{'main_changes_from_original_recipe': ['연어를 닭가...
1,"{'user_allergy_ingredients': [], 'user_dislike...","{'_id': '6761069a846f9e5eb9761021', 'title': '...",{'main_changes_from_original_recipe': ['🍇 포도주스...
2,"{'user_allergy_ingredients': [], 'user_dislike...","{'_id': '6761069a846f9e5eb9760836', 'title': '...",{'main_changes_from_original_recipe': ['🥣 양파와 ...


## 실제로 데이터 만들기 - gpt-4o-mini

In [4]:
from langchain_teddynote import logging
# set_enable=False 로 지정하면 추적을 하지 않습니다.
logging.langsmith("랭체인 튜토리얼 프로젝트", set_enable=False)

LangSmith 추적을 하지 않습니다.


In [1]:
import nest_asyncio
import asyncio
import pandas as pd
import sys
sys.path.append('../')  # 상위 디렉토리의 src 폴더를 경로에 추가
from src.recipe_change_origin import generate_recipe

# nest_asyncio로 이미 실행 중인 루프에서 중첩 실행 허용
nest_asyncio.apply()

# 동시 요청 제한과 재시도 설정
MAX_CONCURRENT_REQUESTS = 3
RETRY_LIMIT = 3

# 데이터 로드
df = pd.read_json('data/combined_dataset.json')

async def generate(df, feature_num):
    df['generation'] = None  # 결과 저장 열 생성

    # Semaphore 생성
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    tasks = []  # 모든 작업을 저장할 리스트

    for i in range(len(df)):
        retry_count = 0
        while retry_count <= RETRY_LIMIT:
            try:
                # Semaphore로 동시 요청 제한
                async with semaphore:
                    print(f"Processing row {i} (Attempt {retry_count + 1})...")
                    result = await generate_recipe(df.iloc[i].recipe_info, df.iloc[i].user_info, feature_num)
                    df.at[i, 'generation'] = result  # 결과 저장
                    break  # 성공하면 반복문 종료
            except Exception as e:
                retry_count += 1
                if retry_count > RETRY_LIMIT:
                    print(f"Failed to process row {i} after {RETRY_LIMIT} retries.")
                    break  # 재시도 초과 시 반복문 종료
                print(f"Error processing row {i}: {e}. Retrying...")
                await asyncio.sleep(10 ** retry_count)  # 재시도 전 대기

        tasks.append(asyncio.sleep(0))  # 리스트에 임시 작업 추가 (추후 확장 가능)

    await asyncio.gather(*tasks)  # 모든 작업 실행

    # 결과 확인
    print(df.head())
    


2024-12-23 02:04:30,363 - recipe_logger - INFO - LLM 초기화 완료: gpt-4o-mini


In [15]:
feature_num = 1  # 사용할 feature_num 설정

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, feature_num))

# 결과 저장
output_file = f'groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 00:58:19,221 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 00:58:19,222 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 00:58:19,226 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 00:58:19,227 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 00:58:19,228 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 0 (Attempt 1)...


2024-12-23 00:58:36,786 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 00:58:36,799 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 00:58:36,799 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 00:58:36,800 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 00:58:36,801 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 00:58:36,802 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 00:58:58,465 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 00:58:58,481 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 00:58:58,482 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 00:58:58,482 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 00:58:58,483 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 00:58:58,484 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 01:08:06,192 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 01:08:06,205 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 01:08:06,205 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:08:06,206 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:08:06,208 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:08:06,209 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 01:08:26,355 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  
0  {'main_changes_from_original_recipe': ['🥗 닭가슴살...  
1  {'main_changes_from_original_recipe': ['🍇 포도주스...  
2  {'main_changes_from_original_recipe': ['🥣 양파는 ...  
3  {'main_changes_from_original_recipe': ['🍋 레몬을 ...  
4  {'main_changes_from_original_recipe': ['🥣 황태채 ..

In [16]:
feature_num = 2  # 사용할 feature_num 설정

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, feature_num))

# 결과 저장
output_file = f'data/groundTruth/groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 01:08:26,403 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 01:08:26,404 - recipe_logger - INFO - langfuse prompt template 생성 완료


Processing row 0 (Attempt 1)...


2024-12-23 01:08:26,632 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:08:26,633 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:08:26,635 - recipe_logger - INFO - LLM 레시피 생성 중...
2024-12-23 01:08:41,887 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 01:08:41,904 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 01:08:41,905 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:08:41,906 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:08:41,907 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:08:41,908 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 01:08:57,318 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 01:08:57,334 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 01:08:57,334 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:08:57,335 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:08:57,336 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:08:57,338 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 01:18:25,574 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 01:18:25,589 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 01:18:25,590 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:18:25,590 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:18:25,591 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:18:25,592 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 01:18:48,963 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  
0  {'main_changes_from_original_recipe': ['연어 대신 ...  
1  {'main_changes_from_original_recipe': ['🍇 포도주스...  
2  {'main_changes_from_original_recipe': ['양파와 감자...  
3  {'main_changes_from_original_recipe': ['레몬을 굵은...  
4  {'main_changes_from_original_recipe': ['소고기를 황..

In [5]:
feature_num = 3  # 사용할 feature_num 설정

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, feature_num))

# 결과 저장
output_file = f'data/groundTruth/groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 02:05:32,392 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 02:05:32,394 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:05:32,397 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:05:32,399 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:05:32,401 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:05:32,402 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 0 (Attempt 1)...


2024-12-23 02:06:02,258 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 02:06:02,270 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 02:06:02,270 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:06:02,271 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:06:02,272 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:06:02,272 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:06:02,273 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 02:06:44,661 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 02:06:44,676 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 02:06:44,676 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:06:44,677 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:06:44,678 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:06:44,679 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:06:44,679 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 02:29:54,097 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 02:29:54,109 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 02:29:54,110 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:29:54,110 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:29:54,111 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:29:54,112 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:29:54,112 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 02:30:32,559 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  
0  {'original_recipe_food_group_composition': [{'...  
1  {'original_recipe_food_group_composition': [{'...  
2  {'original_recipe_food_group_composition': [{'...  
3  {'original_recipe_food_group_composition': [{'...  
4  {'original_recipe_food_group_composition': [{'..

## 실제로 데이터 만들기 - groundTruth

In [1]:
from langchain_teddynote import logging
# set_enable=False 로 지정하면 추적을 하지 않습니다.
logging.langsmith("랭체인 튜토리얼 프로젝트", set_enable=False)

LangSmith 추적을 하지 않습니다.


In [1]:
import nest_asyncio
import asyncio
import pandas as pd
import sys
sys.path.append('../')  # 상위 디렉토리의 src 폴더를 경로에 추가
from src.recipe_change_origin import generate_recipe

# nest_asyncio로 이미 실행 중인 루프에서 중첩 실행 허용
nest_asyncio.apply()

# 동시 요청 제한과 재시도 설정
MAX_CONCURRENT_REQUESTS = 3
RETRY_LIMIT = 3

async def generate(df, feature_num):
    df['groundTruth'] = None  # 결과 저장 열 생성

    # Semaphore 생성
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    tasks = []  # 모든 작업을 저장할 리스트

    for i in range(len(df)):
        retry_count = 0
        while retry_count <= RETRY_LIMIT:
            try:
                # Semaphore로 동시 요청 제한
                async with semaphore:
                    print(f"Processing row {i} (Attempt {retry_count + 1})...")
                    result = await generate_recipe(df.iloc[i].recipe_info, df.iloc[i].user_info, feature_num)
                    df.at[i, 'groundTruth'] = result  # 결과 저장
                    break  # 성공하면 반복문 종료
            except Exception as e:
                retry_count += 1
                if retry_count > RETRY_LIMIT:
                    print(f"Failed to process row {i} after {RETRY_LIMIT} retries.")
                    break  # 재시도 초과 시 반복문 종료
                print(f"Error processing row {i}: {e}. Retrying...")
                await asyncio.sleep(10 ** retry_count)  # 재시도 전 대기

        tasks.append(asyncio.sleep(0))  # 리스트에 임시 작업 추가 (추후 확장 가능)

    await asyncio.gather(*tasks)  # 모든 작업 실행

    # 결과 확인
    print(df.head())
    


2024-12-23 02:34:54,275 - recipe_logger - INFO - LLM 초기화 완료: gpt-4o


In [3]:
# 데이터 로드
feature_num = 1  # 사용할 feature_num 설정

file_path = f'data/groundTruth/groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, feature_num))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 01:41:53,218 - recipe_logger - INFO - json 출력 파서 초기화 완료.


Processing row 0 (Attempt 1)...


2024-12-23 01:41:53,775 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:41:53,949 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:41:54,124 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:41:54,125 - recipe_logger - INFO - LLM 레시피 생성 중...
2024-12-23 01:42:08,735 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 01:42:08,749 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 01:42:08,749 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:42:08,750 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:42:08,751 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:42:08,752 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 01:42:22,475 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 01:42:22,489 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 01:42:22,489 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:42:22,490 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:42:22,491 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:42:22,492 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 01:51:16,075 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 01:51:16,089 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 01:51:16,089 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:51:16,090 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:51:16,091 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:51:16,091 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 01:51:37,171 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'main_changes_from_original_recipe': ['🥗 닭가슴살...   
1  {'main_changes_from_original_recipe': ['🍇 포도주스...   
2  {'main_changes_from_original_recipe': ['🥣 양파는 ...   
3  {'main_changes_from_original_recipe': ['🍋 레몬을 ...   
4  {'main_changes_from_original_recipe': ['🥣 황

In [4]:
# 데이터 로드
feature_num = 2  # 사용할 feature_num 설정

file_path = f'data/groundTruth/groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, feature_num))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 01:51:37,215 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 01:51:37,215 - recipe_logger - INFO - langfuse prompt template 생성 완료


Processing row 0 (Attempt 1)...


2024-12-23 01:51:37,434 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:51:37,438 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:51:37,444 - recipe_logger - INFO - LLM 레시피 생성 중...
2024-12-23 01:51:54,021 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 01:51:54,034 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 01:51:54,035 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:51:54,036 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:51:54,037 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:51:54,038 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 01:52:20,182 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 01:52:20,197 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 01:52:20,197 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:52:20,199 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:52:20,199 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 01:52:20,200 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 02:02:13,266 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 02:02:13,281 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 02:02:13,282 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:02:13,282 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:02:13,283 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:02:13,285 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 02:02:31,620 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'main_changes_from_original_recipe': ['연어 대신 ...   
1  {'main_changes_from_original_recipe': ['🍇 포도주스...   
2  {'main_changes_from_original_recipe': ['양파와 감자...   
3  {'main_changes_from_original_recipe': ['레몬을 굵은...   
4  {'main_changes_from_original_recipe': ['소고기

In [2]:
# 데이터 로드
feature_num = 3  # 사용할 feature_num 설정

file_path = f'data/groundTruth/groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, feature_num))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 02:35:06,124 - recipe_logger - INFO - json 출력 파서 초기화 완료.


Processing row 0 (Attempt 1)...


2024-12-23 02:35:06,752 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:35:06,957 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:35:07,133 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:35:07,313 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 02:35:07,314 - recipe_logger - INFO - LLM 레시피 생성 중...
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"detail":"Forbidden"}')
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"detail":"Forbidden"}')
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST

Processing row 1 (Attempt 1)...


Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"detail":"Forbidden"}')
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"detail":"Forbidden"}')
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"detail":"Forbidden"}')
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: htt

...

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"detail":"Forbidden"}')
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"detail":"Forbidden"}')
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"detail":"Forbidden"}')
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: htt

Processing row 34 (Attempt 1)...


Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"detail":"Forbidden"}')
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"detail":"Forbidden"}')
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"detail":"Forbidden"}')
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: htt

                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'original_recipe_food_group_composition': [{'...   
1  {'original_recipe_food_group_composition': [{'...   
2  {'original_recipe_food_group_composition': [{'...   
3  {'original_recipe_food_group_composition': [{'...   
4  {'original_recipe_food_group_composition': 

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"detail":"Forbidden"}')
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"detail":"Forbidden"}')
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"detail":"Forbidden"}')


In [61]:
df_cp = df.copy().reset_index(drop=True)

In [67]:
feature_eval_result

{'main_changes_from_original_recipe': ['연어를 닭가슴살로 변경했어요! 🐟➡️🍗',
  '발사믹식초를 레몬즙으로 바꿨어요! 🍇➡️🍋',
  '어린잎채소는 그대로 두었어요! 🥗✨',
  '조리 과정을 간단하게 정리했어요! 📋✂️'],
 'reason_for_changes': ['닭가슴살은 필수 재료로 사용해야 하니 연어 대신 넣었어요! 🍗💪',
  '발사믹식초 대신 레몬즙을 사용하면 상큼함을 유지할 수 있어요! 🍋💖',
  '어린잎채소는 신선한 식감을 주기 때문에 그대로 두었어요! 🥗🌱',
  '조리 난이도가 중급인 사용자를 위해 과정을 간단하게 정리했어요! 🛠️✨'],
 'recipe_cooking_order': ['닭가슴살은 구워서 깍둑썰기 해주세요.',
  '구운 닭가슴살에 후춧가루와 레몬즙으로 마리네이드 해주세요.',
  '어린잎채소는 찬물에 담궈 물기를 빼주세요.',
  '레몬즙과 올리브오일을 섞어 드레싱을 만들어주세요.',
  '드레싱에 마리네이드한 닭가슴살과 어린잎채소를 섞어 접시에 담아주세요.'],
 'recipe_cooking_time': '약 20분',
 'recipe_difficulty': '중급',
 'recipe_ingredients': ['닭가슴살(150g)',
  '레몬(1/4개)',
  '레몬즙(50g)',
  '어린잎채소(30g)',
  '후춧가루(0.01g)',
  '올리브오일(20g)'],
 'recipe_menu_name': '닭가슴살 샐러드',
 'recipe_tips': ['드레싱에 소금을 추가하지 않아도 레몬과 올리브오일의 조화로 맛있어요! 🍋✨',
  '닭가슴살을 미리 구워두면 더 간편하게 만들 수 있어요! 🔥🍗'],
 'recipe_type': '샐러드',
 'unchanged_parts_and_reasons': ['어린잎채소는 신선한 식감을 주기 때문에 변경하지 않았어요! 🥗🌿',
  '후춧가루는 간단한 양념으로 요리의 풍미를 살려주기 때문에 그대로 두었어요! 🌶️👌']}

In [58]:
user_info

{'user_allergy_ingredients': [],
 'user_dislike_ingredients': [],
 'user_spicy_level': '3단계',
 'user_cooking_level': '중급',
 'user_owned_ingredients': ['닭가슴살', '두부', '브로콜리'],
 'user_basic_seasoning': ['소금', '후추', '올리브유'],
 'must_use_ingredients': ['닭가슴살']}

KeyError: "Input to PromptTemplate is missing variables {'recipe_info', 'user_info'}.  Expected: ['recipe_info', 'recipe_keyIngredients_tasty', 'user_info'] Received: ['recipe_keyIngredients_tasty']\nNote: if you intended {recipe_info} to be part of the string and not a variable, please escape it with double curly braces like: '{{recipe_info}}'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT"

In [138]:
import asyncio
import time
import sys
sys.path.append('../')  # 상위 디렉토리의 src 폴더를 경로에 추가

from src.recipe_change_origin import langfuse_tracking, get_system_prompt, choose_feature, ChangeRecipe, RecipeChangeBalanceNutrition 
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from src.logger import logger_eval
import pandas as pd
import asyncio

llm = ChatOpenAI(
        model="gpt-4o",
        temperature=0.0,
        max_tokens=1000,
        timeout=40
    )
logger_eval.info("LLM 초기화 완료.")

async def generate_test_dataset(prompt, df, new_column, batch_size=5, max_concurrent_tasks=3):
    prompt = get_system_prompt(prompt)
    llm_chain = prompt | llm | StrOutputParser()

    logger_eval.info("LLM 레시피 생성 중...")
    semaphore = asyncio.Semaphore(max_concurrent_tasks)  # 동시 실행 제한

    async def process_task(i, langfuse_handler):
        async with semaphore:
            for _ in range(3):  # 최대 3번 재시도
                try:
                    result = await llm_chain.ainvoke(
                        input={"user_info": df.user_info[i], "recipe_info": df.recipe_info[i]},
                        config={"callbacks": [langfuse_handler]},
                    )
                    return result
                except Exception as e:
                    if "rate limit" in str(e).lower():
                        logger_eval.warning(f"Rate limit hit for index {i}. Retrying...")
                        await asyncio.sleep(0.5)  # 대기 후 재시도
                    else:
                        logger_eval.error(f"Error at index {i}: {e}")
                        return e  # 재시도 중 에러 발생 시 반환
            logger_eval.error(f"Max retries exceeded for index {i}")
            return None

    for start in range(0, len(df), batch_size):
        end = start + batch_size
        langfuse_handler = langfuse_tracking()
        tasks = [process_task(i, langfuse_handler) for i in range(start, min(end, len(df)))]
        results = await asyncio.gather(*tasks, return_exceptions=True)

        for i, result in enumerate(results):
            if result is None or isinstance(result, Exception):
                logger_eval.error(f"Error at index {start + i}: {result}")
            else:
                df.at[start + i, new_column] = result

        # 요청 사이 지연 추가 (토큰 사용량 분산)
        await asyncio.sleep(0.1)  # 100ms 지연

    logger_eval.info("LLM 레시피 생성 완료")
    return df


In [139]:
# user_info, recipe_info가 있는 dataset -> 결과는 langfuse 2024.12.18 오전 9:9~9:10분꺼부터 보기
df = pd.read_json('data/combined_dataset.json')

# 비동기 함수 실행
df_tasty = await generate_test_dataset("find_keyIngredients_tasty", df, "expected_output")
df_tasty.describe()

,user_info,recipe_info
count,35,35
unique,10,35
top,"{'user_allergy_ingredients': [], 'user_dislike_ingredients': [], 'user_spicy_level': '3단계', 'user_cooking_level': '중급', 'user_owned_ingredients': ['닭가슴살', '두부', '브로콜리'], 'user_basic_seasoning': ['소금', '후추', '올리브유'], 'must_use_ingredients': ['닭가슴살']}","{'_id': '67610699846f9e5eb975e532', 'title': '연어샐러드', 'type_key': '샐러드', 'method_key': '회', 'servings': '1인분', 'cooking_time': '', 'difficulty': '', 'ingredients': ['연어(150g)', '레몬(1/4개)', '발사믹식초(50g)', '어린잎채소(30g)', '후춧가루(0.01g)', '올리브오일(20g)'], 'cooking_steps': ['1. 연어는 깍둑썰기한다.', '2. 썰어 놓은 연어는 후춧가루와 레몬으로 마리네이드한다.', '3. 어린잎은 찬물에 담궈둔다.', '4. 담궈 놓은 어린잎을 체에 받쳐 물기를 뺀다.', '5. 레몬과 올리브오일을 섞는다.', '6. ?번에 발사믹소스를 넣고 연어 샐러드 양념을 만들고, 접시에 연어와 물기를 뺀 어린잎을 담는다.'], 'tips': ['발사믹소스에 레몬과 올리브오일을 섞어 별도의 소금을 넣지 않을 수 있어요.']}"
freq,4,1


In [121]:
# user, recipe 정보만 저장
# df_tasty.iloc[:,:-1].to_csv("data/test_userRecipe.csv", index=False)

# user, recipe, 맛분석 결과 저장
df_tasty.to_csv("data/test_userRecipeTasty.csv", index=False)

In [124]:
df_tasty[:2]

,user_info,recipe_info,recipe_keyIngredients_tasty
0,"{'user_allergy_ingredients': [], 'user_dislike_ingredients': [], 'user_spicy_level': '3단계', 'user_cooking_level': '중급', 'user_owned_ingredients': ['닭가슴살', '두부', '브로콜리'], 'user_basic_seasoning': ['소금', '후추', '올리브유'], 'must_use_ingredients': ['닭가슴살']}","{'_id': '67610699846f9e5eb975e532', 'title': '연어샐러드', 'type_key': '샐러드', 'method_key': '회', 'servings': '1인분', 'cooking_time': '', 'difficulty': '', 'ingredients': ['연어(150g)', '레몬(1/4개)', '발사믹식초(50g)', '어린잎채소(30g)', '후춧가루(0.01g)', '올리브오일(20g)'], 'cooking_steps': ['1. 연어는 깍둑썰기한다.', '2. 썰어 놓은 연어는 후춧가루와 레몬으로 마리네이드한다.', '3. 어린잎은 찬물에 담궈둔다.', '4. 담궈 놓은 어린잎을 체에 받쳐 물기를 뺀다.', '5. 레몬과 올리브오일을 섞는다.', '6. ?번에 발사믹소스를 넣고 연어 샐러드 양념을 만들고, 접시에 연어와 물기를 뺀 어린잎을 담는다.'], 'tips': ['발사믹소스에 레몬과 올리브오일을 섞어 별도의 소금을 넣지 않을 수 있어요.']}","### 레시피 맛 설명\n연어샐러드는 신선한 연어의 부드러운 식감과 어린잎채소의 아삭함이 조화를 이루는 요리입니다. 발사믹식초와 레몬의 상큼한 맛이 연어의 풍미를 돋우며, 올리브오일이 전체적으로 부드러운 맛을 더해줍니다. 후춧가루는 약간의 매콤함을 더해 맛의 균형을 잡아줍니다.\n\n### 핵심 재료와 비핵심 재료\n- **핵심 재료:**\n - **연어:** 이 요리의 주재료로, 신선한 해산물의 맛과 부드러운 식감을 제공합니다.\n - **어린잎채소:** 신선한 채소의 아삭한 식감과 상큼한 맛을 더해줍니다.\n - **발사믹식초:** 샐러드 드레싱의 기본이 되는 재료로, 상큼하고 달콤한 맛을 제공합니다.\n - **레몬:** 연어의 비린 맛을 잡아주고 상큼함을 더해줍니다.\n\n- **비핵심 재료:**\n - **후춧가루:** 약간의 매콤함을 더해주는 역할을 하지만, 없어도 큰 맛의 변화는 없습니다.\n - **올리브오일:** 드레싱의 부드러움을 더해주지만, 다른 오일로 대체 가능할 수 있습니다.\n\n### 재료 변경 및 이유\n- **연어 → 닭가슴살:** 사용자가 반드시 사용해야 하는 재료인 닭가슴살을 활용합니다. 닭가슴살은 연어와는 다른 식감과 맛을 제공하지만, 단백질 공급원으로서의 역할을 유지할 수 있습니다. 닭가슴살은 부드럽고 담백한 맛을 가지고 있어, 발사믹식초와 레몬 드레싱과 잘 어울립니다.\n- **어린잎채소, 발사믹식초, 레몬, 후춧가루, 올리브오일:** 이 재료들은 그대로 유지합니다. 닭가슴살과도 잘 어울리며, 샐러드의 신선함과 드레싱의 맛을 유지하는 데 필수적입니다.\n\n이러한 변경은 사용자의 재료 제약 조건을 충족하면서도 원래 레시피의 맛과 조화를 최대한 유지하려는 목적을 가지고 있습니다."
1,"{'user_allergy_ingredients': [], 'user_dislike_ingredients': ['토마토', '오이'], 'user_spicy_level': '2단계', 'user_cooking_level': '초급', 'user_owned_ingredients': ['계란', '감자', '스팸'], 'user_basic_seasoning': ['간장', '설탕', '후추'], 'must_use_ingredients': ['계란']}","{'_id': '6761069a846f9e5eb9761021', 'title': '포도우유젤리 만들기', 'type_key': '디저트', 'method_key': '끓이기', 'servings': '3인분', 'cooking_time': '30분 이내', 'difficulty': '초급', 'ingredients': ['포도주스(300g)', '설탕(50g)', '레몬즙(1작은술)', '젤라틴(가루)(15g)', '물(젤라틴)(5-6큰술)', '우유(200g)', '생크림(100g)', '설탕(30g)', '바닐라오일(2-3방울)', '젤라틴(가루)(15g)', '물(젤라틴)(5-6큰술)'], 'cooking_steps': ['먼저 찬물에 가루 젤라틴을 불려 주세요. 가루 젤라틴을 사용하는 법은 가루젤라틴 양의 5-6배의 찬물에 불려서 사용하시면 됩니다. 불리는 시간은 10분 정도이면 됩니다. 전 2가지 젤리를 만들 것이기 때문에 2개분의 젤라틴을 찬물에 불렸습니다.', '냄비에 포도주스(또는 기타 주스)와 설탕을 넣고 약한 불에 올려 설탕이 녹을 정도로 데워 주세요. 끓이시면 안됩니다.', '찬물에 불린 젤라틴을 넣고 잘 저어 완전히 녹여 주세요. 간혹 젤라틴이 덩어리져 잘 안 풀리다는 분이 계신데 이럴 때에는 체에 한 번 걸러 주세요.마지막에 레몬쥬스를 넣고 잘 섞어 주세요.', '틀에 1/2 정도만 붓고 냉장고에서 1시간 이상 굳혀 주세요.', '파나코타는 포도젤리가 다 굳은 후 만드셔야 합니다. 냄비에 우유, 생크림, 설탕, 바닐에오일을 넣고 중간 불에서 데워 주세요. 설탕이 녹이기 위한 것이니 끓이지는 마세요.', '역시 찬물에 불린 젤라틴을 넣고 잘 저어 젤라틴을 잘 녹여 주세요.', '체에 한 번 걸은 후 상온에서 약간 식혀 포도젤리 위에 부어 냉장고에서 1시간 이상 굳혀 주세요.'], 'tips': ['젤리 틀에서 잘 꺼내는 법은? 젤리는 특성상 따뜻한 물이 닿으면 살짝 녹습니다. 이 성질을 이용하여서 굳은 젤리 그릇의 바닥을 뜨거운 물에 담갔다가 뒤집으면 아주 깨끗하게 잘 떨어집니다. 그러나 너무 오래 담가 놓으면 예쁜 모양이 나오지 않는답니다.']}","### 레시피 맛 설명\n포도우유젤리는 상큼한 포도 맛과 부드러운 우유의 조화가 돋보이는 디저트입니다. 포도주스의 상큼함과 레몬즙의 산미가 젤리의 상쾌한 맛을 더해주며, 우유와 생크림의 부드러움이 파나코타 층에서 느껴집니다. 바닐라오일은 은은한 향을 더해 전체적인 맛을 풍부하게 만들어 줍니다.\n\n### 핵심 재료와 비핵심 재료\n- **핵심 재료:**\n - **포도주스:** 젤리의 주된 맛을 결정하는 재료로, 상큼한 포도 맛을 제공합니다.\n - **젤라틴:** 젤리의 형태를 잡아주는 필수적인 재료입니다.\n - **우유와 생크림:** 파나코타 층의 부드러움과 크리미한 맛을 제공합니다.\n\n- **비핵심 재료:**\n - **설탕:** 단맛을 추가하지만, 다른 감미료로 대체 가능하여 비핵심으로 분류됩니다.\n - **레몬즙:** 산미를 더하지만, 없어도 큰 맛의 변화는 없습니다.\n - **바닐라오일:** 향을 더해주지만, 필수적이지는 않습니다.\n\n### 재료 변경 제안\n사용자의 정보와 제약 조건에 따라 레시피를 변경할 필요는 없습니다. 사용자는 특정 알레르기나 싫어하는 재료가 없으며, 포도우유젤리 레시피에 포함된 재료들은 모두 사용 가능합니다. 또한, 사용자가 반드시 사용해야 하는 재료로 '계란'이 있지만, 이 레시피에 계란을 추가하는 것은 맛과 질감에 큰 영향을 미칠 수 있으므로, 레시피의 본질을 유지하기 위해 계란을 추가하지 않는 것이 좋습니다.\n\n따라서, 레시피는 그대로 유지하며, 사용자가 소유한 기본 조미료(간장, 설탕, 후추)는 이미 레시피에 포함되어 있어 추가적인 변경이 필요하지 않습니다."


### dataset에 레시피 영양성분 분석 결과 추가

In [ ]:
df = pd.read_json('data/combined_dataset.json')
df_cp = df.copy().reset_index(drop=True)
final_df = await generate_test_dataset("generate_food_group_ratio", df_cp, "expected_output")
final_df.describe()

In [ ]:
final_df.to_csv("data/test_groupRatio.csv", index=False)

### dataset에 레시피 영양성분 분석 결과 추가

In [ ]:
df = pd.read_json('data/combined_dataset.json')
df_cp = df.copy().reset_index(drop=True)
final_df = await generate_test_dataset("generate_food_group_ratio", df_cp, "expected_output")
final_df.describe()

# 데이터 가독성을 높여 출력하는 함수

In [ ]:

def display_columns_with_format(df, col1, col2):
    for i, row in df.iterrows():
        print(f"Row {i + 1}:")
        print(f"--- {col1} ---")
        print(row[col1])  # 딕셔너리나 JSON 형태를 보기 좋게 출력
        print(f"\n--- {col2} ---")
        print(row[col2])  # 마크다운 텍스트 그대로 출력
        print("\n" + "=" * 40 + "\n")

# 함수 호출
display_columns_with_format(df, "recipe_info", "original_recipe_food_group_composition")
